<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/Training%20the%20LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')
!pip install ultralytics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.6 MB/s eta 0:00:00


# LSTM training

In [7]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ─── STEP 1: Load keypoints from Drive ───
drive_keypoints = "/content/drive/MyDrive/shoplifting/keypoints"
local_keypoints = "/content/keypoints"

if not os.path.exists(local_keypoints):
    print("Copying keypoints from Drive...")
    shutil.copytree(drive_keypoints, local_keypoints)
    print("Done.")

print("Loading keypoint files...")
X, y = [], []

for file in os.listdir(local_keypoints):
    if file.endswith('.json'):
        with open(os.path.join(local_keypoints, file)) as f:
            data = json.load(f)
        X.append(data['keypoints'])
        y.append(data['label'])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"Loaded: {X.shape} — Labels: {y.shape}")
print(f"Shoplifting: {int(y.sum())} | Normal: {int((y==0).sum())}")

# ─── STEP 2: Normalize ───
X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / (X_std + 1e-8)

# ─── STEP 3: Train/val split ───
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)}")

# ─── STEP 4: Dataset ───
class KeypointDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = KeypointDataset(X_train, y_train)
val_ds = KeypointDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

# ─── STEP 5: LSTM Model ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing: {device}")

model = ShopliftingLSTM().to(device)

# Class weights for imbalance
pos_weight = torch.tensor([621/416]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

# ─── STEP 6: Training ───
EPOCHS = 30
best_val_acc = 0

print("\nTraining...\n")
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct += ((preds > 0.5) == yb).sum().item()
        total += len(yb)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_loss += criterion(preds, yb).item()
            val_correct += ((preds > 0.5) == yb).sum().item()
            val_total += len(yb)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_dl):.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/vigiq_best.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

# ─── STEP 7: Evaluation ───
model.load_state_dict(torch.load('/content/vigiq_best.pth'))
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_preds.extend((preds > 0.5).cpu().numpy())
        all_labels.extend(yb.numpy())

print("\n=== RESULTS ===")
print(classification_report(all_labels, all_preds,
      target_names=['Normal', 'Shoplifting']))
print(f"Best Val Accuracy: {best_val_acc:.4f}")

# ─── STEP 8: Save to Drive ───
torch.save(model.state_dict(), '/content/drive/MyDrive/shoplifting/vigiq_best.pth')
np.save('/content/drive/MyDrive/shoplifting/X_mean.npy', np.array([X_mean]))
np.save('/content/drive/MyDrive/shoplifting/X_std.npy', np.array([X_std]))
print("\nModel saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading keypoint files...
Loaded: (1037, 50, 34) — Labels: (1037,)
Shoplifting: 416 | Normal: 621

Train: 829 | Val: 208

Using: cuda

Training...

Epoch 01/30 | Train Loss: 0.6779 | Train Acc: 0.5766 | Val Acc: 0.6010
  ✓ Best model saved (val_acc: 0.6010)
Epoch 02/30 | Train Loss: 0.6640 | Train Acc: 0.5983 | Val Acc: 0.6010
Epoch 03/30 | Train Loss: 0.6599 | Train Acc: 0.5983 | Val Acc: 0.5817
Epoch 04/30 | Train Loss: 0.6576 | Train Acc: 0.5838 | Val Acc: 0.6010
Epoch 05/30 | Train Loss: 0.6491 | Train Acc: 0.6333 | Val Acc: 0.6058
  ✓ Best model saved (val_acc: 0.6058)
Epoch 06/30 | Train Loss: 0.6435 | Train Acc: 0.6490 | Val Acc: 0.6442
  ✓ Best model saved (val_acc: 0.6442)
Epoch 07/30 | Train Loss: 0.6414 | Train Acc: 0.6273 | Val Acc: 0.6779
  ✓ Best model saved (val_acc: 0.6779)
Epoch 08/30 | Train Loss: 0.6273 | Train Acc: 0.6634 | Val Acc: 0.6250

In [ ]:
# ─── KEYPOINT EXTRACTION v2 — NORMALIZED ───
#
# What changed from v1:
#
# v1 — Raw pixel coordinates
#   - Saved x,y positions as they appear on screen
#   - Problem: same theft motion looks different depending on
#     how far person is from camera
#   - Close person = large numbers, far person = small numbers
#   - LSTM got confused, couldn't generalize
#
# v2 — Body-relative normalization
#   - Find body center = midpoint between left and right hip
#   - Find body scale = distance from hip center to shoulder center
#   - Subtract center from every keypoint (makes position relative)
#   - Divide by scale (removes camera distance effect)
#   - Now wrist moving to hip looks the same from any distance
#   - LSTM sees motion pattern, not screen position
#
# Result: model should generalize better across different
# camera angles and distances
#
# Output saved to: keypoints_normalized/

In [8]:
import cv2
import numpy as np
import os
import json
import shutil
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

# ─── Load model ───
model = YOLO("yolo11n-pose.pt")

dataset_paths = [
    ("/content/shoplifting_data/shoplifting/shoplifting", 1),
    ("/content/shoplifting_data/shoplifting/normal", 0),
    ("/content/shoplifting_data/shoplifting_1/Shop DataSet/shop lifters", 1),
    ("/content/shoplifting_data/shoplifting_1/Shop DataSet/non shop lifters", 0),
]

output_dir = "/content/keypoints_normalized"
os.makedirs(output_dir, exist_ok=True)

SAMPLE_EVERY = 3
MAX_FRAMES = 50

# YOLO11 pose keypoint indices
LEFT_HIP  = 11
RIGHT_HIP = 12
LEFT_SHOULDER  = 5
RIGHT_SHOULDER = 6

def normalize_keypoints(kps):
    """
    Normalize keypoints relative to body center and scale.
    kps shape: (17, 2) — x,y for each keypoint
    """
    # Body center = midpoint of hips
    left_hip  = kps[LEFT_HIP]
    right_hip = kps[RIGHT_HIP]
    center = (left_hip + right_hip) / 2

    # Body scale = distance from hip center to shoulder center
    left_shoulder  = kps[LEFT_SHOULDER]
    right_shoulder = kps[RIGHT_SHOULDER]
    shoulder_center = (left_shoulder + right_shoulder) / 2
    scale = np.linalg.norm(shoulder_center - center)

    if scale < 1e-6:
        scale = 1.0  # avoid division by zero

    # Normalize: subtract center, divide by scale
    normalized = (kps - center) / scale
    return normalized.flatten().tolist()

def extract_keypoints_normalized(video_path):
    cap = cv2.VideoCapture(video_path)
    keypoint_sequence = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % SAMPLE_EVERY == 0:
            results = model(frame, verbose=False)

            if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
                kps = results[0].keypoints.xy[0].cpu().numpy()  # (17, 2)

                if kps.shape[0] == 17:
                    normalized = normalize_keypoints(kps)
                    keypoint_sequence.append(normalized)
                else:
                    keypoint_sequence.append([0] * 34)
            else:
                keypoint_sequence.append([0] * 34)

            if len(keypoint_sequence) >= MAX_FRAMES:
                break

        frame_idx += 1

    cap.release()
    return keypoint_sequence

# ─── Run extraction ───
total = 0
skipped = 0

for folder_path, label in dataset_paths:
    files = [f for f in os.listdir(folder_path) if f.endswith('.mp4')]
    print(f"\nProcessing: {folder_path} — {len(files)} clips")

    for filename in files:
        video_path = os.path.join(folder_path, filename)
        sequence = extract_keypoints_normalized(video_path)

        if len(sequence) < 10:
            skipped += 1
            continue

        while len(sequence) < MAX_FRAMES:
            sequence.append([0] * 34)
        sequence = sequence[:MAX_FRAMES]

        output = {
            "file": filename,
            "label": label,
            "keypoints": sequence
        }

        out_path = os.path.join(output_dir, f"{label}_{total}.json")
        with open(out_path, 'w') as f:
            json.dump(output, f)

        total += 1
        if total % 50 == 0:
            print(f"  Processed {total} clips...")

print(f"\nExtraction done. {total} clips. {skipped} skipped.")

# ─── Save to Drive ───
drive_output = "/content/drive/MyDrive/shoplifting/keypoints_normalized"
if os.path.exists(drive_output):
    shutil.rmtree(drive_output)
shutil.copytree(output_dir, drive_output)
print(f"Saved to Drive → {drive_output}")
print(f"Total files: {len(os.listdir(drive_output))}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: /content/shoplifting_data/shoplifting/shoplifting — 92 clips
  Processed 50 clips...

Processing: /content/shoplifting_data/shoplifting/normal — 90 clips
  Processed 100 clips...
  Processed 150 clips...

Processing: /content/shoplifting_data/shoplifting_1/Shop DataSet/shop lifters — 324 clips
  Processed 200 clips...
  Processed 250 clips...
  Processed 300 clips...
  Processed 350 clips...
  Processed 400 clips...
  Processed 450 clips...
  Processed 500 clips...

Processing: /content/shoplifting_data/shoplifting_1/Shop DataSet/non shop lifters — 531 clips
  Processed 550 clips...
  Processed 600 clips...
  Processed 650 clips...
  Processed 700 clips...
  Processed 750 clips...
  Processed 800 clips...
  Processed 850 clips...
  Processed 900 clips...
  Processed 950 clips...
  Processed 1000 clips...

Extraction done. 1037 clips. 0 skipped.
S

# retraining LSTM on normalized data to get higher shoplifting recall

In [9]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shutil
from google.colab import drive

drive.mount('/content/drive')

# ─── STEP 1: Load normalized keypoints ───
drive_keypoints = "/content/drive/MyDrive/shoplifting/keypoints_normalized"
local_keypoints = "/content/keypoints_normalized"

if not os.path.exists(local_keypoints):
    print("Copying from Drive...")
    shutil.copytree(drive_keypoints, local_keypoints)
    print("Done.")

print("Loading keypoint files...")
X, y = [], []

for file in os.listdir(local_keypoints):
    if file.endswith('.json'):
        with open(os.path.join(local_keypoints, file)) as f:
            data = json.load(f)
        X.append(data['keypoints'])
        y.append(data['label'])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"Loaded: {X.shape} — Labels: {y.shape}")
print(f"Shoplifting: {int(y.sum())} | Normal: {int((y==0).sum())}")

# ─── STEP 2: Normalize ───
X_mean = X.mean()
X_std = X.std()
X = (X - X_mean) / (X_std + 1e-8)

# ─── STEP 3: Train/val split ───
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {len(X_train)} | Val: {len(X_val)}")

# ─── STEP 4: Dataset ───
class KeypointDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = KeypointDataset(X_train, y_train)
val_ds   = KeypointDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=32)

# ─── STEP 5: Improved LSTM ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing: {device}")

model = ShopliftingLSTM().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=4, factor=0.5
)

# ─── STEP 6: Training ───
EPOCHS = 50
best_val_acc = 0

print("\nTraining...\n")
for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        correct += ((preds > 0.5) == yb).sum().item()
        total += len(yb)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            val_loss += criterion(preds, yb).item()
            val_correct += ((preds > 0.5) == yb).sum().item()
            val_total += len(yb)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_dl):.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/vigiq_best_v2.pth')
        print(f"  ✓ Best model saved (val_acc: {val_acc:.4f})")

# ─── STEP 7: Evaluation ───
model.load_state_dict(torch.load('/content/vigiq_best_v2.pth'))
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_preds.extend((preds > 0.5).cpu().numpy())
        all_labels.extend(yb.numpy())

print("\n=== RESULTS v2 ===")
print(classification_report(all_labels, all_preds,
      target_names=['Normal', 'Shoplifting']))
print(f"Best Val Accuracy: {best_val_acc:.4f}")

# ─── STEP 8: Save to Drive ───
torch.save(model.state_dict(),
           '/content/drive/MyDrive/shoplifting/vigiq_best_v2.pth')
np.save('/content/drive/MyDrive/shoplifting/X_mean_v2.npy', np.array([X_mean]))
np.save('/content/drive/MyDrive/shoplifting/X_std_v2.npy',  np.array([X_std]))
print("\nModel saved to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading keypoint files...
Loaded: (1037, 50, 34) — Labels: (1037,)
Shoplifting: 416 | Normal: 621

Train: 829 | Val: 208

Using: cuda

Training...

Epoch 01/50 | Train Loss: 0.6802 | Train Acc: 0.5983 | Val Acc: 0.6010
  ✓ Best model saved (val_acc: 0.6010)
Epoch 02/50 | Train Loss: 0.6578 | Train Acc: 0.5983 | Val Acc: 0.6010
Epoch 03/50 | Train Loss: 0.6268 | Train Acc: 0.6297 | Val Acc: 0.6250
  ✓ Best model saved (val_acc: 0.6250)
Epoch 04/50 | Train Loss: 0.6367 | Train Acc: 0.6429 | Val Acc: 0.6779
  ✓ Best model saved (val_acc: 0.6779)
Epoch 05/50 | Train Loss: 0.6120 | Train Acc: 0.6876 | Val Acc: 0.6779
Epoch 06/50 | Train Loss: 0.6048 | Train Acc: 0.6972 | Val Acc: 0.6875
  ✓ Best model saved (val_acc: 0.6875)
Epoch 07/50 | Train Loss: 0.5918 | Train Acc: 0.6888 | Val Acc: 0.6827
Epoch 08/50 | Train Loss: 0.5894 | Train Acc: 0.7238 | Val Acc: 0.7163